In [ ]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor



In [ ]:
connection_string = (
    "mssql+pyodbc://@localhost/mlb?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine = create_engine(connection_string)

query = """
select * from mlb.dbo.fact_hitter_pitcher_matchup_model_features
"""

df = pd.read_sql(query, engine)
print(df.shape)
df.head()

(24609, 448)


,gamePk,game_date,season,hitter_id,hitter_name,hitter_position,hitter_team_id,hitter_team_name,pitcher_id,pitcher_name,...,pitcher_avg_chase_rate_last_10,pitcher_avg_zone_rate_last_10,pitcher_avg_velocity_last_10,pitcher_avg_spin_rate_last_10,pitcher_avg_sl_whiff_rate_last_10,pitcher_avg_ff_whiff_rate_last_10,pitcher_prev_whiff_rate,pitcher_prev_csw_rate,pitcher_prev_chase_rate,pitcher_strikeOuts
0,778032,2025-05-06,2025,672695,Geraldo Perdomo,SS,109,Arizona Diamondbacks,656849,David Peterson,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
1,777013,2025-07-25,2025,666181,Will Benson,LF,113,Cincinnati Reds,641793,Zack Littell,...,0.282231,0.534879,87.684055,1827.276940,0.078336,0.122787,0.112360,0.235955,0.306122,2
2,776712,2025-08-16,2025,672580,Maikel Garcia,3B,118,Kansas City Royals,680732,Sean Burke,...,0.320924,0.493231,88.397858,2492.036148,0.124928,0.119645,0.102273,0.306818,0.324324,3
3,776152,2025-09-27,2025,669369,Bryce Johnson,CF,135,San Diego Padres,694851,Andrew Hoffmann,...,0.263626,0.448639,90.804374,1635.216126,0.230159,0.117464,0.090909,0.181818,0.181818,0
4,777731,2025-05-28,2025,671289,Tyler Freeman,RF,115,Colorado Rockies,571510,Matthew Boyd,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8


In [ ]:
# to a list of all the columns in a txt file 
with open("columns.txt", "w") as f:
    for col in df.columns:
        f.write(col + "\n")

In [12]:
# define features and targets
# X = features & y = target

target = "pitcher_strikeOuts"

drop_cols = [
    "gamePk",
    "game_date",
    "hitter_name",
    "pitcher_name",
    "hitter_position",
    "hitter_team_name",
    "pitcher_team_name"
]


leakage_cols = [
    "hitter_strikeOuts",
    "pitches_seen_vs_pitcher",
    "swings_vs_pitcher",
    "whiffs_vs_pitcher",
    "called_strikes_vs_pitcher",
    "matchup_whiff_rate",
    "matchup_called_strike_rate",
    "matchup_csw_rate"
]

X = df.drop(columns=drop_cols + leakage_cols + [target])
y = df[target]

print(X.shape)
print(y.shape)


(24609, 432)
(24609,)


In [13]:
important_features = [
        # ===== Pitcher form =====
    "pitcher_avg_k_last_3",
    "pitcher_avg_k_last_5",
    "pitcher_avg_k_last_10",
    "pitcher_prev_k",
        # ===== Opportunity =====
    "pitcher_avg_bf_last_3",
    "pitcher_avg_ip_last_3",
    "pitcher_avg_pitches_last_3",
    "pitcher_avg_outs_last_3",
        # ===== Skill =====
    "pitcher_avg_whiff_rate_last_3",
    "pitcher_avg_whiff_rate_last_5",
    "pitcher_avg_whiff_rate_last_10",
    "pitcher_avg_velocity_last_3",
    "pitcher_avg_spin_rate_last_3",
    "pitcher_avg_putaway_rate_last_3",
        # ===== Context =====
    "pitcher_throws",
    "hitter_stand"
    ""
]

X = X[important_features]
print(X.shape)


(24609, 16)


In [14]:
train_df = df[df["season"] == 2025]
test_df = df[df["season"] == 2026]

X_train = train_df[important_features]
y_train = train_df[target]

X_test = test_df[important_features]
y_test = test_df[target]

print(X_train.shape, X_test.shape)

(20905, 16) (3704, 16)


In [15]:
X_train = pd.get_dummies(X_train, columns=["pitcher_throws", "hitter_stand"], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=["pitcher_throws", "hitter_stand"], drop_first=True)

# Align columns (VERY IMPORTANT)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [16]:
model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [17]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[5.292556  5.2666264 5.8953276 5.077209  3.779169  6.0871763 1.5693507
 5.6125827 1.2733473 5.622337 ]


In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 1.7902448177337646
RMSE: 2.2149879906460113


In [ ]:
results = test_df.copy()

results["predicted_strikeouts"] = y_pred

results[[
    "pitcher_name",
    "pitcher_team_name",   
    "pitcher_strikeOuts",
    "predicted_strikeouts"
]].head(20)

,pitcher_name,pitcher_team_name,pitcher_strikeOuts,predicted_strikeouts
18,Chad Patrick,Milwaukee Brewers,4,5.292556
30,Brandon Williamson,Cincinnati Reds,4,5.266626
40,Connelly Early,Boston Red Sox,4,5.895328
46,Will Warren,New York Yankees,6,5.077209
47,Freddy Peralta,New York Mets,7,3.779169
49,Gavin Williams,Cleveland Guardians,10,6.087176
51,Anthony Bender,Miami Marlins,1,1.569351
60,Albert Suárez,Baltimore Orioles,2,5.612583
62,Jordan Romano,Los Angeles Angels,2,1.273347
79,Logan Gilbert,Seattle Mariners,7,5.622337


In [ ]:
pitcher_preds = results.groupby(
    ["gamePk", "pitcher_name", "pitcher_team_name"]  # 👈 added
).agg(
    actual_K=("pitcher_strikeOuts", "first"),
    predicted_K=("predicted_strikeouts", "mean")
).reset_index()

pitcher_preds.head(50)

,gamePk,pitcher_name,pitcher_team_name,actual_K,predicted_K
0,822753,Brad Lord,Washington Nationals,2,3.091576
1,822753,Cole Henry,Washington Nationals,3,1.457871
2,822753,Michael McGreevy,St. Louis Cardinals,1,4.682560
3,822753,Miles Mikolas,Washington Nationals,3,4.596870
4,822753,PJ Poulin,Washington Nationals,0,1.473283
5,822753,Riley O'Brien,St. Louis Cardinals,0,1.216035
6,822754,Cade Cavalli,Washington Nationals,3,3.977152
7,822754,Cole Henry,Washington Nationals,1,1.537610
8,822754,George Soriano,St. Louis Cardinals,3,1.237577
9,822754,Matthew Liberatore,St. Louis Cardinals,6,4.854645
